In [1]:
import ReportMailer as rm

# Install a conda package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install xlsxwriter

# Reporting Period #
* PD - yesterday
* PW - previous week
* PM - previous month
* PQ - previous quarter
* PY - previous calendar year
* YTD year to date
* WTD current week to date
* TODAY today
* P90 - previous 90days

In [2]:
acronyms = [ "PD", "PW", "PM", "PQ", "PY", "YTD", "WTD", "TODAY", "P90"]
friendly = [ "Prior Day", "Prior Week", "Prior Month", "Prior Quarter", "Prior Year", "Year to Date", "Week to Date", "Today", "Previous 90 days"]

def toPeriodFriendly(INTERVAL):
    for n in range(0, len(acronyms)):
      if INTERVAL==acronyms[n]:
         return friendly[n]
    return "Unknown Period"

class StopExecution(Exception):
    def _render_traceback_(self):
        pass

raise StopExecution

def exit(): raise StopExecution

In [3]:
import os
import pandas as pd

path = "/home/gbadmin/jupy-notebooks/Reports/inqueue"
dir_list = os.listdir(path)

if len(dir_list)>0:
  filename = "inqueue/{}".format(dir_list[0])
  print(filename)

  rep = pd.read_json(filename, lines=True)

  for _ , row in rep.iterrows():    
    _INTERVAL = row['interval']
    _TYPE = row['type']
    if (_TYPE=='ALL'):
       _APPUSER  = row['appuser']
       _CCLIST = row['cc'];
       print(_INTERVAL)
       print(_TYPE)
       print(_CCLIST)
       # delete the file
       os.remove(filename)
    else:
      print("Non-Rollup type")
      exit()
       
else:
  print("NOTHING TO DO")
  exit()



inqueue/rollup-report-20260928_133045-charlie.json
PQ
ALL
['11']


#_INTERVAL = "PY"
_INTERVAL = 'YYYYMM202307'

In [4]:
from pandas import ExcelWriter
from pandas import ExcelFile
import xlsxwriter
import datetime

def createSpreadsheetAndMailIt(collections, reportname, recipients, cc, subject, body):
  now = datetime.datetime.now().strftime('%Y-%m-%d_%H%M%S')
  filename = reportname + "-" + now + ".xlsx"
  #writer = ExcelWriter(filename)
  writer = ExcelWriter(filename, engine='xlsxwriter')
  workbook = writer.book  
  print("Writing dataframe to Excel file {0}".format(filename))
  for collection in collections:
    print("Writing {} to spreadsheet".format(collection["name"]))
    collection["dataframe"].to_excel(writer, sheet_name=collection["name"], index=False)
    worksheet = writer.sheets[collection["name"]]
    if 'colwidths' in collection:
      colwidths = collection['colwidths']
      print("colwidths={}".format(colwidths))
      for n in range(0, len(colwidths)):
        col = str(chr(65 + n))
        worksheet.set_column('{}:{}'.format(col,col), colwidths[n])
    else:
      worksheet.set_column('A:A', 30)
  writer.close()
  #for recipient in recipients: 
  rm.mailer(recipients, subject, body, filename, cc)
  print("Excel/Email Done!")

In [5]:
import psycopg2 as pg
import pandas.io.sql as psql
import pandas as pd
import GuestbookDbConnect as gdb

conn = gdb.guestbookDbConnect()

Welcome to Jupyter Notebook.  You are connected to the Opportunity House guestbook database!


In [6]:
import ReportTimerange as rtr

if _INTERVAL.startswith("YYYYMM"):
  yyyymm = _INTERVAL[6:12]
  print(yyyymm)
  yyyy = int(yyyymm[0:4])
  mm = int(yyyymm[4:6])
  print("{} {}".format(yyyy,mm))
  trange = rtr.mmyyyy(mm,yyyy)
else:
  trange = rtr.timerange(_INTERVAL)
print(trange)

collections = []
summaries = []

collection = {}
collection["name"] = "Outline"

df = pd.DataFrame({"Report Tab":[ \
              "Summary", \
              "Distinct Clients", \
              "Services", \
              "Housing", \
              "Incarcerations", \
              "Gender", \
              "Ethnicity", \
              "Veterans", \
              "New Clients"
                                 ], \
                   "Description":[ \
              "summary totals for the report period", \
              "a count of individuals served by sub-period within the period.", \
              "breakdown by service type and sub-period within the period.", \
              "breakdown by housing conditions and sub-period within the period.", \
              "clients reporting incarceration in previous 90 day period.", \
              "breakdown by gender and sub-period within the period.", \
              "breakdown by ethnicity and sub-period within the period.", \
              "breakdown by veteran status and sub-period within the period.", \
              "a list of clients with first visit occurring in the report period."
                                 ]})

collection["dataframe"] = df 
collection["colwidths"] = [30, 100]
collections.append(collection)

df.head(1000)

[datetime.datetime(2026, 4, 1, 0, 0), datetime.datetime(2026, 6, 30, 23, 59, 59, 999999), 'YYYY-MM/DD WW']


,Report Tab,Description
0,Summary,summary totals for the report period
1,Distinct Clients,a count of individuals served by sub-period wi...
2,Services,breakdown by service type and sub-period withi...
3,Housing,breakdown by housing conditions and sub-period...
4,Incarcerations,clients reporting incarceration in previous 90...
5,Gender,breakdown by gender and sub-period within the ...
6,Ethnicity,breakdown by ethnicity and sub-period within t...
7,Veterans,breakdown by veteran status and sub-period wit...
8,New Clients,a list of clients with first visit occurring i...


## Total distinct clients in report period ##

In [7]:
summary = {}
summary["name"] = "Total distinct clients in report period"

query = "SELECT min(timestamp)AS start, max(timestamp) AS end, \
         COUNT(DISTINCT person_id) AS clients FROM guestbook_personsnapshot \
         WHERE timestamp BETWEEN '{}' AND '{}'".format(trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

summary["count"] = data.iloc[0,2]
summaries.append(summary)
print(summaries)

data.head(100)

SELECT min(timestamp)AS start, max(timestamp) AS end,          COUNT(DISTINCT person_id) AS clients FROM guestbook_personsnapshot          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}]


/tmp/ipykernel_156235/625775520.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,start,end,clients
0,2026-04-02 11:46:47.470849+00:00,2026-06-30 16:04:37.800938+00:00,307


# Distinct clients by sub-interval within the report period #
* Statistics focused on the breadth of individuals that are served during the time period.

In [8]:
collection = {}
collection["name"] = "Distinct Clients"

query = "SELECT to_char(timestamp,'{}') AS period,  \
         COUNT(DISTINCT person.idperson) AS clients \
         FROM guestbook_personsnapshot snapshot \
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY period".format(trange[2], trange[0], trange[1])
print(query)

data = pd.read_sql(query, conn)
#data.head(1000)

persons = data.pivot_table('clients', index=['period']).fillna(0).astype(int).reset_index('period')

collection["dataframe"] = persons 
collection["colwidths"] = [30, 20]
collections.append(collection)
persons.head(1000)

SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,           COUNT(DISTINCT person.idperson) AS clients          FROM guestbook_personsnapshot snapshot          JOIN guestbook_person person ON person.idperson=snapshot.person_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY period


/tmp/ipykernel_156235/3547540159.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,period,clients
0,2026-04/02 14,35
1,2026-04/03 14,22
2,2026-04/06 14,38
3,2026-04/07 14,38
4,2026-04/08 14,34
...,...,...
56,2026-06/24 25,42
57,2026-06/25 26,40
58,2026-06/26 26,39
59,2026-06/29 26,52


## Total Services delivered in report period ##

In [9]:
query = "SELECT \
         left(service.name, strpos(service.name, '/') - 1) AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot \
         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
         WHERE timestamp BETWEEN '{}' AND '{}'  AND service.points<=0 \
         GROUP BY servicename \
         ORDER BY total desc".format(trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)


for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = data.iloc[n,0] + " services delivered."
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
print(summaries)
data.head(1000)

SELECT          left(service.name, strpos(service.name, '/') - 1) AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot          JOIN guestbook_service service ON service.idservice=servicerequest.service_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'  AND service.points<=0          GROUP BY servicename          ORDER BY total desc
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}, {'name': 'Meal services delivered.', 'count': np.int64(2451)}, {'name': 'Showe services delivered.', 'count': np.int64(917)}, {'name': 'Clothin services delivered.', 'count': np.int64(404)}, {'name': 'Laundr services delivered.', 'count': np.int64(254)}, {'name': 'Nurs services delivered.', 'count': np.int64(34)}, {'name': 'Housing Assistanc services delivered.', 'count': np.int64(18)}, {'name': 'Toiletrie servi

/tmp/ipykernel_156029/142895627.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,servicename,total
0,Meal,2451
1,Showe,917
2,Clothin,404
3,Laundr,254
4,Nurs,34
5,Housing Assistanc,18
6,Toiletrie,9
7,Group Clas,3
8,Compute,2
9,Pastor Consultatio,2


# Services delivered by sub-interval within the report period #

In [10]:
collection = {}
collection["name"] = "Services"

#query = "SELECT to_char(timestamp,'{}') AS period, \
#         service.name AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
#         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot \
#         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
#         WHERE timestamp BETWEEN '{}' AND '{}' \
#         GROUP BY period, servicename".format(trange[2], trange[0], trange[1])

query = "SELECT to_char(timestamp,'{}') AS period, \
         left(service.name, strpos(service.name, '/') - 1) AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot \
         JOIN guestbook_service service ON service.idservice=servicerequest.service_id \
         WHERE timestamp BETWEEN '{}' AND '{}' AND service.points<=0 \
         GROUP BY period, servicename".format(trange[2], trange[0], trange[1])



print(query)

data = pd.read_sql(query, conn)

services = data.pivot_table('total', index=['period'], columns='servicename').fillna(0).astype(int).reset_index('period')
collection["dataframe"] = services 
collection["colwidths"] = [30, 20, 20, 20, 20, 20, 20, 20]
collections.append(collection)

services.head(1000)

SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,          left(service.name, strpos(service.name, '/') - 1) AS servicename, COUNT(*) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_personservicerequest servicerequest ON servicerequest.connection_id=snapshot.idsnapshot          JOIN guestbook_service service ON service.idservice=servicerequest.service_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND service.points<=0          GROUP BY period, servicename


/tmp/ipykernel_154478/896303.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


servicename,period,Clothin,Compute,Group Clas,Housing Assistanc,Laundr,Meal,Nurs,Pastor Consultatio,Showe,Toiletrie
0,2026-04/02 14,9,0,0,0,1,35,0,0,15,0
1,2026-04/03 14,4,0,0,0,5,22,0,0,8,0
2,2026-04/06 14,3,0,0,0,2,37,1,0,12,0
3,2026-04/07 14,8,0,0,0,6,36,2,0,9,0
4,2026-04/08 14,5,0,0,0,3,34,0,0,11,0
...,...,...,...,...,...,...,...,...,...,...,...
56,2026-06/24 25,6,0,0,0,4,40,1,0,13,0
57,2026-06/25 26,3,0,0,0,2,39,0,0,15,0
58,2026-06/26 26,6,0,0,3,6,36,0,0,14,3
59,2026-06/29 26,3,0,1,0,8,51,0,0,19,0


# Housing totals within the report period#

In [11]:
query = "SELECT  \
         left(housing.name, strpos(housing.name, '(') - 1)  AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
         GROUP BY hresponse".format(trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = " Days spent in " + data.iloc[n,0] 
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
print(summaries)

data.head(1000)

SELECT           left(housing.name, strpos(housing.name, '(') - 1)  AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot          JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND prompt_id=8          GROUP BY hresponse
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}, {'name': 'Meal services delivered.', 'count': np.int64(2451)}, {'name': 'Showe services delivered.', 'count': np.int64(917)}, {'name': 'Clothin services delivered.', 'count': np.int64(404)}, {'name': 'Laundr services delivered.', 'count': np.int64(254)}, {'name': 'Nurs services delivered.', 'count': np.int64(34)}, {'name': 'Housing Assistanc services delivered.', 'count': np.int64(18)}, {'name': 'Toiletrie services delivered.', 'count': np.int64(9)}, {'name': 'Group Clas se

/tmp/ipykernel_154478/3219514869.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,hresponse,total
0,Unsheltered,1701
1,Sheltered,70
2,Stable housing,302
3,Jail or priso,6
4,Unstable housing,461


# Housing by City within the report period#

In [12]:
query = "SELECT  \
         city.name AS cresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_cityresponse city ON city.idcity=survey.object_id \
         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=53 \
         GROUP BY cresponse ORDER BY total desc".format(trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = " Nights spent in " + data.iloc[n,0] 
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
print(summaries)

data.head(1000)

SELECT           city.name AS cresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot          JOIN guestbook_cityresponse city ON city.idcity=survey.object_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND prompt_id=53          GROUP BY cresponse ORDER BY total desc
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}, {'name': 'Meal services delivered.', 'count': np.int64(2451)}, {'name': 'Showe services delivered.', 'count': np.int64(917)}, {'name': 'Clothin services delivered.', 'count': np.int64(404)}, {'name': 'Laundr services delivered.', 'count': np.int64(254)}, {'name': 'Nurs services delivered.', 'count': np.int64(34)}, {'name': 'Housing Assistanc services delivered.', 'count': np.int64(18)}, {'name': 'Toiletrie services delivered.', 'count': np.int64(9)}, {'name': 'Group Clas services delivered.', 'count': np.

/tmp/ipykernel_154478/1462714343.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,cresponse,total
0,Concord,2084
1,Other,17
2,Kannapolis,8


# Housing by sub-interval within the report period#
* Answers to the prompt 'Where did you spend last night?'

In [13]:
collection = {}
collection["name"] = "Housing"

#query = "SELECT to_char(timestamp,'{}') AS period, \
#         housing.name AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
#         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
#         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
#         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
#         GROUP BY period, hresponse".format(trange[2], trange[0], trange[1])

query = "SELECT to_char(timestamp,'{}') AS period, \
         left(housing.name, strpos(housing.name, '(') - 1) AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
         GROUP BY period, hresponse".format(trange[2], trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

housing = data.pivot_table('total', index=['period'], columns='hresponse').fillna(0).astype(int).reset_index('period')
#housing = housing.fillna(0)
collection["dataframe"] = housing  
collection["colwidths"] = [30, 30, 30, 30, 30, 30]
collections.append(collection)
housing.head(1000)

SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,          left(housing.name, strpos(housing.name, '(') - 1) AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot          JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND prompt_id=8          GROUP BY period, hresponse


/tmp/ipykernel_154478/325975016.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


hresponse,period,Jail or priso,Sheltered,Stable housing,Unsheltered,Unstable housing
0,2026-04/02 14,0,0,4,32,1
1,2026-04/03 14,0,1,1,18,4
2,2026-04/06 14,0,0,3,26,9
3,2026-04/07 14,0,2,7,25,5
4,2026-04/08 14,0,0,7,20,8
...,...,...,...,...,...,...
56,2026-06/24 25,0,0,4,34,3
57,2026-06/25 26,0,0,5,16,20
58,2026-06/26 26,0,5,4,27,4
59,2026-06/29 26,0,0,1,47,3


# Housing by City and sub-interval within the report period#

In [14]:
collection = {}
collection["name"] = "Housing by City"

#query = "SELECT to_char(timestamp,'{}') AS period, \
#         housing.name AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
#         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
#         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
#         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
#         GROUP BY period, hresponse".format(trange[2], trange[0], trange[1])

query = "SELECT to_char(timestamp,'{}') AS period, \
         city.name AS cresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_cityresponse city ON city.idcity=survey.object_id \
         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=53 \
         GROUP BY period, cresponse".format(trange[2], trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

city = data.pivot_table('total', index=['period'], columns='cresponse').fillna(0).astype(int).reset_index('period')
#housing = housing.fillna(0)
collection["dataframe"] = city  
collection["colwidths"] = [30, 30, 30, 30]
collections.append(collection)
city.head(1000)

SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,          city.name AS cresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot          JOIN guestbook_cityresponse city ON city.idcity=survey.object_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND prompt_id=53          GROUP BY period, cresponse


/tmp/ipykernel_154478/933571557.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


cresponse,period,Concord,Kannapolis,Other
0,2026-04/02 14,30,0,0
1,2026-04/03 14,21,0,0
2,2026-04/06 14,31,0,0
3,2026-04/07 14,27,0,1
4,2026-04/08 14,27,0,0
...,...,...,...,...
56,2026-06/24 25,35,0,0
57,2026-06/25 26,37,0,0
58,2026-06/26 26,31,1,1
59,2026-06/29 26,45,0,0


# Clients (by name) Reporting Unsheltered Housing #
* more than once in the report period

In [15]:
collection = {}
collection["name"] = "Unsheltered_by_Name"

unsheltered = 5
threshold = 1

#query = "SELECT to_char(timestamp,'{}') AS period, \
#         housing.name AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
#         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
#         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
#         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
#         GROUP BY period, hresponse".format(trange[2], trange[0], trange[1])

query = "SELECT person.firstname, person.lastname, person.aliasname, \
         COUNT(*) AS unsheltered_nights FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 AND object_id={} \
         GROUP BY person.idperson ORDER BY COUNT(*) desc".format(trange[0], trange[1], unsheltered)

print(query)

data = pd.read_sql(query, conn)
data.head(100)

#housing = data.pivot_table('total', index=['period'], columns='hresponse').fillna(0).astype(int).reset_index('period')
collection["dataframe"] = data 
collection["colwidths"] = [30, 30, 30, 30, 30, 30]
collections.append(collection)
data.head(1000)

SELECT person.firstname, person.lastname, person.aliasname,          COUNT(*) AS unsheltered_nights FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot          JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND prompt_id=8 AND object_id=5          GROUP BY person.idperson ORDER BY COUNT(*) desc


/tmp/ipykernel_154478/4085064638.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,firstname,lastname,aliasname,unsheltered_nights
0,KYLE,GORDON,0000KYLE_GORDON,48
1,RUSSELL,SPRINKLE,0000RUSSELL_SPRINKLE,48
2,JULIAN,CARTER,JULIANC,46
3,DANAH,AL SADAT,0000DANAH_AL SADAT,46
4,ROBERT,FRESKE,0000ROBERT_F,45
...,...,...,...,...
225,TANYA,PARKER,0000TANYA_PARKER,1
226,DEVYN,HARDY,0000DEVYN_HARDY,1
227,CAMDEN,WILHIT,0000CAMDEN_WILHIT,1
228,ASHLEY,ABBOTT,0000ASHLEY_ABBOTT,1


# Clients Reporting Incarceration in last 90 days #

In [16]:
collection = {}
collection["name"] = "Incarcerations"


#query = "SELECT to_char(timestamp,'{}') AS period, \
#         housing.name AS hresponse, COUNT(*) AS total FROM guestbook_personsnapshot snapshot\
#         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
#         JOIN guestbook_housingresponse housing ON housing.idhousing=survey.object_id \
#         WHERE timestamp BETWEEN '{}' AND '{}' AND prompt_id=8 \
#         GROUP BY period, hresponse".format(trange[2], trange[0], trange[1])

query = "SELECT person.idperson, person.firstname, person.lastname\
         FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot \
         WHERE survey.object_id=2 AND timestamp BETWEEN '{}' AND '{}' AND prompt_id=4 \
         GROUP BY person.idperson, person.firstname, person.lastname \
         ORDER BY person.idperson".format(trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

#housing = data.pivot_table('total', index=['period'], columns='hresponse').fillna(0).astype(int).reset_index('period')
collection["dataframe"] = data 
collection["colwidths"] = [30, 30, 30]
collections.append(collection)
data.head(1000)

SELECT person.idperson, person.firstname, person.lastname         FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_personsurvey survey ON survey.connection_id=snapshot.idsnapshot          WHERE survey.object_id=2 AND timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999' AND prompt_id=4          GROUP BY person.idperson, person.firstname, person.lastname          ORDER BY person.idperson


/tmp/ipykernel_154478/3461077659.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,idperson,firstname,lastname
0,141,KEITH,GRISSOM
1,478,JULIAN,CARTER
2,493,JOHNNY,CHAVIS
3,1483,DAVID,QUERY
4,1539,JOSEPH,THRIFT
5,1549,JAZMINE,CAMPBELL
6,2250,JASON,MITCHELL
7,2292,RUSSELL,SPRINKLE
8,2453,ANDREW,EDDINGS
9,2788,ASHELY,LACY


# Gender totals in the report period #

In [17]:
query = "SELECT  \
         gender.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_genderresponse gender ON gender.idgender=person.gender_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY response".format(trange[0], trange[1])

print(query)
#summaries = []

data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = "Gender -" + data.iloc[n,0] 
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
print(summaries)

data.head(100)

SELECT           gender.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_genderresponse gender ON gender.idgender=person.gender_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY response
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}, {'name': 'Meal services delivered.', 'count': np.int64(2451)}, {'name': 'Showe services delivered.', 'count': np.int64(917)}, {'name': 'Clothin services delivered.', 'count': np.int64(404)}, {'name': 'Laundr services delivered.', 'count': np.int64(254)}, {'name': 'Nurs services delivered.', 'count': np.int64(34)}, {'name': 'Housing Assistanc services delivered.', 'count': np.int64(18)}, {'name': 'Toiletrie services delivered.', 'count': np.int64(9)}, {'name': 'Group Clas services delivered.', 'count': np.int64(3)}, {'name'

/tmp/ipykernel_154478/179570838.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,response,total
0,Female,78
1,Male,174
2,Unknown,55


# Gender by sub-interval within the report period #

In [18]:
collection = {}
collection["name"] = "Gender"

query = "SELECT to_char(timestamp,'{}') AS period, \
         gender.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_genderresponse gender ON gender.idgender=person.gender_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY period, response".format(trange[2], trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

gender = data.pivot_table('total', index=['period'], columns='response').fillna(0).astype(int).reset_index('period')
collection["dataframe"] = gender
collection["colwidths"] = [30, 20, 20, 20]
#collection["colwidths"] = []
collections.append(collection)
gender.head(1000)

SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,          gender.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_genderresponse gender ON gender.idgender=person.gender_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY period, response


/tmp/ipykernel_154478/2926783295.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


response,period,Female,Male,Unknown
0,2026-04/02 14,9,25,1
1,2026-04/03 14,4,18,0
2,2026-04/06 14,6,30,2
3,2026-04/07 14,7,28,3
4,2026-04/08 14,6,27,1
...,...,...,...,...
56,2026-06/24 25,9,32,1
57,2026-06/25 26,8,32,0
58,2026-06/26 26,9,28,2
59,2026-06/29 26,12,38,2


# Ethnicity totals within the report period #

In [19]:
query = "SELECT  \
         ethnicity.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_ethnicityresponse ethnicity ON ethnicity.idethnicity=person.ethnicity_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY response".format(trange[0], trange[1])

print(query)
#summaries = []

data = pd.read_sql(query, conn)
for n in range(0, data.shape[0]):  
  summary = {}
  summary["name"]  = "Ethnicity -" + data.iloc[n,0] 
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
print(summaries)
data.head(1000)

SELECT           ethnicity.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_ethnicityresponse ethnicity ON ethnicity.idethnicity=person.ethnicity_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY response
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}, {'name': 'Meal services delivered.', 'count': np.int64(2451)}, {'name': 'Showe services delivered.', 'count': np.int64(917)}, {'name': 'Clothin services delivered.', 'count': np.int64(404)}, {'name': 'Laundr services delivered.', 'count': np.int64(254)}, {'name': 'Nurs services delivered.', 'count': np.int64(34)}, {'name': 'Housing Assistanc services delivered.', 'count': np.int64(18)}, {'name': 'Toiletrie services delivered.', 'count': np.int64(9)}, {'name': 'Group Clas services delivered.', 'count': np.

/tmp/ipykernel_154478/2817021490.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,response,total
0,Asian,5
1,Black,147
2,Hispanic,7
3,Native American,1
4,Other,5
5,Unknown,2
6,White,140


# Ethnicity by sub-interval within the report period #

In [20]:
collection = {}
collection["name"] = "Ethnicity"

query = "SELECT to_char(timestamp,'{}') AS period, \
         ethnicity.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_ethnicityresponse ethnicity ON ethnicity.idethnicity=person.ethnicity_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY period, response".format(trange[2], trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)

ethnicity = data.pivot_table('total', index=['period'], columns='response').fillna(0).astype(int).reset_index('period')
collection["dataframe"] = ethnicity 
collection["colwidths"] = [30, 20, 20, 20, 20, 20, 20, 20]
collections.append(collection)
ethnicity.head(1000)

SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,          ethnicity.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_ethnicityresponse ethnicity ON ethnicity.idethnicity=person.ethnicity_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY period, response


/tmp/ipykernel_154478/292920172.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


response,period,Asian,Black,Hispanic,Native American,Other,Unknown,White
0,2026-04/02 14,2,12,2,0,3,0,16
1,2026-04/03 14,1,9,1,0,1,0,10
2,2026-04/06 14,1,16,2,0,1,0,18
3,2026-04/07 14,1,16,1,0,3,0,17
4,2026-04/08 14,1,17,1,0,2,0,13
...,...,...,...,...,...,...,...,...
56,2026-06/24 25,2,18,0,0,1,0,21
57,2026-06/25 26,1,12,0,0,1,0,26
58,2026-06/26 26,2,11,1,0,1,0,24
59,2026-06/29 26,2,22,0,0,1,0,27


# Veteran status totals within the report period#

In [21]:
query = "SELECT  \
         yesno.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_yesnoresponse yesno ON yesno.idyesno=person.veteran_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY response \
         ORDER BY response desc".format(trange[0], trange[1])

print(query)
#summaries = []

data = pd.read_sql(query, conn)

for n in range(0, data.shape[0]):  
  summary = {}
  if data.iloc[n,0]=="Yes":
    summary["name"] = "Veterans"
  elif data.iloc[n,0]=="No":
    summary["name"] = "Non-veterans"
  else:
    summary["name"] = "Veteran status Unknown"
  summary["count"] = data.iloc[n,1]
  summaries.append(summary)
print(summaries)

data.head(1000)

SELECT           yesno.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_yesnoresponse yesno ON yesno.idyesno=person.veteran_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY response          ORDER BY response desc
[{'name': 'Total distinct clients in report period', 'count': np.int64(307)}, {'name': 'Meal services delivered.', 'count': np.int64(2451)}, {'name': 'Showe services delivered.', 'count': np.int64(917)}, {'name': 'Clothin services delivered.', 'count': np.int64(404)}, {'name': 'Laundr services delivered.', 'count': np.int64(254)}, {'name': 'Nurs services delivered.', 'count': np.int64(34)}, {'name': 'Housing Assistanc services delivered.', 'count': np.int64(18)}, {'name': 'Toiletrie services delivered.', 'count': np.int64(9)}, {'name': 'Group Clas services delivered.', 'c

/tmp/ipykernel_154478/2044650481.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,response,total
0,Yes,15
1,Unknown,58
2,No,234


# Veteran Status by sub-interval within the report period#

In [22]:
collection = {}
collection["name"] = "Veterans"

query = "SELECT to_char(timestamp,'{}') AS period, \
         yesno.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot\
         JOIN guestbook_person person ON person.idperson=snapshot.person_id \
         JOIN guestbook_yesnoresponse yesno ON yesno.idyesno=person.veteran_id \
         WHERE timestamp BETWEEN '{}' AND '{}' \
         GROUP BY period, response".format(trange[2], trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)
#data.head(1000)



veterans  = data.pivot_table('total', index=['period'], columns='response').fillna(0).astype(int).reset_index('period')
veterans.columns = ['period', 'non-veteran', 'unknown','veteran']
collection["dataframe"] = veterans 
collection["colwidths"] = [30, 30, 30, 30]
collections.append(collection)
veterans.head(1000)


SELECT to_char(timestamp,'YYYY-MM/DD WW') AS period,          yesno.name AS response, COUNT(DISTINCT person.idperson) AS total FROM guestbook_personsnapshot snapshot         JOIN guestbook_person person ON person.idperson=snapshot.person_id          JOIN guestbook_yesnoresponse yesno ON yesno.idyesno=person.veteran_id          WHERE timestamp BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          GROUP BY period, response


/tmp/ipykernel_154478/1470108471.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,period,non-veteran,unknown,veteran
0,2026-04/02 14,33,1,1
1,2026-04/03 14,22,0,0
2,2026-04/06 14,35,2,1
3,2026-04/07 14,34,3,1
4,2026-04/08 14,31,1,2
...,...,...,...,...
56,2026-06/24 25,39,1,2
57,2026-06/25 26,38,0,2
58,2026-06/26 26,36,2,1
59,2026-06/29 26,48,2,2


# New Clients within the report period#
* Clients first appearing in the report interval

In [23]:
collection = {}
collection["name"] = "New Clients"

query = "SELECT firstname, lastname, aliasname, timelinestarttime::date AS startdate FROM guestbook_person \
         WHERE timelinestarttime BETWEEN '{}' AND '{}' \
         ORDER BY timelinestarttime asc".format(trange[0], trange[1])

print(query)

data = pd.read_sql(query, conn)
collection["dataframe"] = data
collection["colwidths"] = [30, 30, 30, 30]
collections.append(collection)

data.head(5000)

SELECT firstname, lastname, aliasname, timelinestarttime::date AS startdate FROM guestbook_person          WHERE timelinestarttime BETWEEN '2026-04-01 00:00:00' AND '2026-06-30 23:59:59.999999'          ORDER BY timelinestarttime asc


/tmp/ipykernel_154478/2192267891.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,firstname,lastname,aliasname,startdate
0,DELONTAJ,SCOTT,0000DELONTAJ_SCOTT,2026-04-02
1,ZEKE,JOHNSON,0000ZEKE_JOHNSON,2026-04-02
2,MERCEDES,RODRIGUEZ,0000MERCEDES_RODRIGUEZ,2026-04-06
3,VANZIADE,DEMORAES,0000VANZIADE_DEMORAES,2026-04-06
4,JAMES,JACKSON,1234JAMESAJACKSON,2026-04-07
...,...,...,...,...
87,DEVYN,HARDY,0000DEVYN_HARDY,2026-06-29
88,CAMDEN,WILHIT,0000CAMDEN_WILHIT,2026-06-29
89,ASHLEY,ABBOTT,0000ASHLEY_ABBOTT,2026-06-29
90,KENNETH,SMITH,0000KENNETH_SMITH,2026-06-29


# Summary Counts #

In [24]:
collection = {}
summ = pd.DataFrame.from_dict(summaries)
collection["name"] = "Summary"
collection["dataframe"] = summ
collection["colwidths"] = [30, 60]
collections.insert(1, collection)
#collections.append(collection)
summ.head(1000)

,name,count
0,Total distinct clients in report period,307
1,Meal services delivered.,2451
2,Showe services delivered.,917
3,Clothin services delivered.,404
4,Laundr services delivered.,254
5,Nurs services delivered.,34
6,Housing Assistanc services delivered.,18
7,Toiletrie services delivered.,9
8,Group Clas services delivered.,3
9,Compute services delivered.,2


In [9]:
_REPORTNAME = _INTERVAL
_SUBJECT = "Opportunity House - Reports for {}".format(toPeriodFriendly(_INTERVAL))
_BODY    = "Spreadsheet (attached) with reports for period {} through {}.".format(trange[0], trange[1])
#_EMAIL_RECIPIENT = ['cprice9739@carolina.rr.com', 'jahood1@yahoo.com']
#_EMAIL_RECIPIENT = ['cprice9739@carolina.rr.com', 'pastor@opphouse.net']
#_EMAIL_RECIPIENT = ['cprice9739@carolina.rr.com']

query = "SELECT idappuser, email FROM guestbook_appuser WHERE idappuser={}".format(_APPUSER)

#print(query)
data = pd.read_sql(query, conn)
_EMAIL_RECIPIENT = []
_EMAIL_RECIPIENT.append(data.iloc[0,1])
_EMAIL_CC = []
for ccid in _CCLIST:
    ccquery = "SELECT idappuser, email FROM guestbook_appuser WHERE idappuser={}".format(ccid)
    data = pd.read_sql(ccquery, conn)
    _EMAIL_CC.append(data.iloc[0,1])
    
print("to: {}".format(_EMAIL_RECIPIENT))
print("cc: {}".format(_EMAIL_CC))

createSpreadsheetAndMailIt(collections, _REPORTNAME, _EMAIL_RECIPIENT, _EMAIL_CC, _SUBJECT, _BODY)

/tmp/ipykernel_156235/2302381856.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)
/tmp/ipykernel_156235/2302381856.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(ccquery, conn)


to: ['holobox@gmail.com']
cc: ['cprice9739@carolina.rr.com']
Writing dataframe to Excel file PQ-2026-09-28_133126.xlsx
Writing Outline to spreadsheet
colwidths=[30, 100]
Writing Distinct Clients to spreadsheet
colwidths=[30, 20]
Emailer Done!
Excel/Email Done!
